In [ ]:
# Setup Env: Dotenv/Pathlib
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# 1. Localização de caminhos e carga de variáveis
load_dotenv(find_dotenv())
PROJECT_ROOT = Path(find_dotenv()).parent
EXTRAIDOS_DIR = PROJECT_ROOT / "data_lake" / "external" / "extraidos"

# 2. Sincronização do Python (WSL/Conda)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 3. Extração e Validação de Credenciais
AWS_ACCESS_KEY_ID     = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN     = os.getenv("AWS_SESSION_TOKEN")
S3_BUCKET             = os.getenv("S3_BUCKET")

# Guard-rail compacto
if not all([AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN, S3_BUCKET]):
    raise EnvironmentError("Falha ao carregar credenciais AWS ou S3_BUCKET do arquivo .env")

print(f"Projeto: {PROJECT_ROOT}\nBucket : {S3_BUCKET}")

In [ ]:
# SparkSession: S3 Config
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("exploracao-e-armazenar-S3")
    .master("local[*]")
    # Configuração de Pacotes e S3A
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)

print(f"Sessão Spark {spark.version} criada com sucesso.")

In [3]:
# Boto3: Gestão de pastas
import boto3

# Inicialize o cliente S3 logo após o Setup de credenciais
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)